# Big Data Engineering — Final Project Part II
Luis Guillermo Rivera Stephens 
Sebastian Tadeo Quiroz Tejeda
Nicolas Navarro Valenzuela

## 1. Create SparkSession

In [1]:
from spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F

kafka_connector   = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
mongodb_connector = "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0"

su = SparkUtils(
    "BigData-Final-Consumer",
    "spark://spark-master:7077",
    spark_packages=f"{kafka_connector},{mongodb_connector}"
)
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4c23877c-f0e4-4c4f-a850-87da82127003;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central


## 2. Schema Definition

In [2]:
transaction_schema = SparkUtils.generate_schema([
    ("transaction_id",   "string"),
    ("customer_id",      "string"),
    ("customer_name",    "string"),
    ("customer_email",   "string"),
    ("customer_country", "string"),
    ("product_id",       "string"),
    ("product_name",     "string"),
    ("category",         "string"),
    ("quantity",         "int"),
    ("unit_price",       "double"),
    ("total_amount",     "double"),
    ("discount_pct",     "double"),
    ("payment_method",   "string"),
    ("status",           "string"),
    ("order_date",       "date"),
    ("order_timestamp",  "timestamp"),
    ("shipping_country", "string"),
    ("shipping_city",    "string"),
    ("warehouse_id",     "int"),
    ("is_returned",      "boolean"),
    ("review_score",     "int"),
    ("review_text",      "string"),
])

## 3. Read Stream from Kafka

In [3]:
raw_stream = (
    su.spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "kafka:9093")
        .option("subscribe", "store-transactions")
        .option("startingOffsets", "latest")
        .option("failOnDataLoss", "false")
        .load()
)

parsed_df = (
    raw_stream
        .selectExpr("CAST(value AS STRING) AS json_str")
        .withColumn("data", F.from_json(F.col("json_str"), transaction_schema))
        .select("data.*")
)

## 4. Transformations

### T1 — Enrichment: derived columns (mirrors pipeline.py logic)

In [4]:
enriched_df = (
    parsed_df
        .withColumn(
            "revenue_after_discount",
            F.round(F.col("total_amount") * (1 - F.col("discount_pct") / 100), 2)
        )
        .filter(
            (F.col("status") != "cancelled") & (F.col("total_amount") > 0)
        )
        .withColumn(
            "transaction_size",
            F.when(F.col("total_amount") > 10000, "large")
             .when(F.col("total_amount") > 3000,  "medium")
             .otherwise("small")
        )
        .withColumn(
            "is_high_value",
            (F.col("total_amount") > 5000) & (F.col("status") == "completed")
        )
)

### T2 — Aggregation: revenue and volume per category using a tumbling window

In [5]:
category_agg_df = (
    enriched_df
        .withWatermark("order_timestamp", "2 minutes")
        .groupBy(
            F.window(F.col("order_timestamp"), "2 minutes", "1 minute"),
            F.col("category"),
            F.col("payment_method")
        )
        .agg(
            F.count("transaction_id").alias("total_transactions"),
            F.round(F.sum("revenue_after_discount"), 2).alias("total_revenue"),
            F.round(F.avg("review_score"), 2).alias("avg_review_score"),
            F.sum(F.col("is_returned").cast("int")).alias("total_returns")
        )
        .withColumn("window_start", F.col("window.start"))
        .withColumn("window_end",   F.col("window.end"))
        .drop("window")
)

### T3 — Aggregation: orders per country (mirrors pipeline.py join logic)

In [6]:
country_agg_df = (
    enriched_df
        .withWatermark("order_timestamp", "2 minutes")
        .groupBy(
            F.window(F.col("order_timestamp"), "2 minutes", "1 minute"),
            F.col("customer_country")
        )
        .agg(
            F.count("*").alias("orders_per_country"),
            F.round(F.sum("revenue_after_discount"), 2).alias("country_revenue"),
            F.round(F.avg("total_amount"), 2).alias("avg_ticket")
        )
        .withColumn("window_start", F.col("window.start"))
        .withColumn("window_end",   F.col("window.end"))
        .drop("window")
)

``` bash
docker run -d \
  --name mongodb-iteso \
  -p 27017:27017 \
  -v mongo_data:/data/db \
  mongo:7.0
```

## 5. MongoDB Sink via foreachBatch


In [7]:
MONGO_URI = "mongodb://mongodb-iteso:27017"
DATABASE  = "store_analytics"

def write_transactions(batch_df, batch_id):
    """Write raw enriched transactions — one document per transaction."""
    if batch_df.isEmpty():
        return
    (
        batch_df.write
            .format("mongodb")
            .option("connection.uri", MONGO_URI)
            .option("database",   DATABASE)
            .option("collection", "transactions")
            .mode("append")
            .save()
    )

def write_category_agg(batch_df, batch_id):
    """Write windowed category aggregations."""
    if batch_df.isEmpty():
        return
    (
        batch_df.write
            .format("mongodb")
            .option("connection.uri", MONGO_URI)
            .option("database",   DATABASE)
            .option("collection", "category_stats")
            .mode("append")
            .save()
    )

def write_country_agg(batch_df, batch_id):
    """Write windowed country aggregations."""
    if batch_df.isEmpty():
        return
    (
        batch_df.write
            .format("mongodb")
            .option("connection.uri", MONGO_URI)
            .option("database",   DATABASE)
            .option("collection", "country_stats")
            .mode("append")
            .save()
    )

## 6. Start Streaming Queries

In [ ]:
for path in ["/opt/spark/work-dir/checkpoints/transactions",
             "/opt/spark/work-dir/checkpoints/category_agg",
             "/opt/spark/work-dir/checkpoints/country_agg"]:
    p = Path(path)
    if p.exists():
        shutil.rmtree(p)

query_transactions = (
    enriched_df.writeStream
        .foreachBatch(write_transactions)
        .outputMode("append")
        .option("checkpointLocation", "/opt/spark/work-dir/checkpoints/transactions")
        .trigger(processingTime="10 seconds")
        .start()
)

query_category = (
    category_agg_df.writeStream
        .foreachBatch(write_category_agg)
        .outputMode("append")
        .option("checkpointLocation", "/opt/spark/work-dir/checkpoints/category_agg")
        .trigger(processingTime="10 seconds")
        .start()
)

query_country = (
    country_agg_df.writeStream
        .foreachBatch(write_country_agg)
        .outputMode("append")
        .option("checkpointLocation", "/opt/spark/work-dir/checkpoints/country_agg")
        .trigger(processingTime="10 seconds")
        .start()
)

print("Streaming queries started. Press Ctrl+C to stop.")
su.spark.streams.awaitAnyTermination()


26/05/11 00:37:36 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/11 00:37:36 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/11 00:37:36 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Streaming queries started. Press Ctrl+C to stop.


StreamingQueryException: [STREAM_FAILED] Query [id = 2aa6501c-aa98-4dcd-9b54-5e10de3d6b89, runId = 780902a9-044a-426e-a857-4d77ead443ed] terminated with exception: Failed to create new KafkaAdminClient SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 2aa6501c-aa98-4dcd-9b54-5e10de3d6b89, runId = 780902a9-044a-426e-a857-4d77ead443ed]
Current Committed Offsets: {}
Current Available Offsets: {}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 ForeachBatchSink, 2aa6501c-aa98-4dcd-9b54-5e10de3d6b89, [checkpointLocation=/opt/spark/work-dir/checkpoints/category_agg], Append
+- ~Project [category#23, payment_method#28, total_transactions#41L, total_revenue#42, avg_review_score#43, total_returns#44L, window_start#75, window_end#77]
   +- ~Project [window#74-T120000ms, category#23, payment_method#28, total_transactions#41L, total_revenue#42, avg_review_score#43, total_returns#44L, window_start#75, window#74-T120000ms.end AS window_end#77]
     

26/05/11 00:37:52 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 14929 milliseconds
26/05/11 00:37:52 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 15504 milliseconds
26/05/11 00:38:41 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000} milliseconds, but spent 11760 milliseconds


## 7. Validation Queries in MongoDB
Run these in `mongosh` after the stream has been running for a few minutes.

```bash
docker exec -it <mongodb_container> mongosh
```

```javascript
use store_analytics

// Check documents arrived
db.transactions.countDocuments()
db.category_stats.countDocuments()
db.country_stats.countDocuments()

// Preview raw transactions
db.transactions.find().limit(3).pretty()
```

### Aggregation pipeline — Top categories by revenue (completed transactions only)
```javascript
db.transactions.aggregate([
  { $match: { status: "completed" } },
  { $group: {
      _id: "$category",
      total_revenue:  { $sum: "$revenue_after_discount" },
      avg_ticket:     { $avg: "$total_amount" },
      total_orders:   { $count: {} },
      avg_review:     { $avg: "$review_score" },
      total_returned: { $sum: { $cond: ["$is_returned", 1, 0] } }
  }},
  { $sort: { total_revenue: -1 } }
])
```

### Aggregation pipeline — Top countries by number of orders
```javascript
db.country_stats.aggregate([
  { $group: {
      _id: "$customer_country",
      total_orders:  { $sum: "$orders_per_country" },
      total_revenue: { $sum: "$country_revenue" }
  }},
  { $sort: { total_orders: -1 } },
  { $limit: 10 }
])
```

In [ ]:
su.spark.stop()